In [1]:
# !pip install "ray[data,train,tune,serve]"
# !pip install -U ipywidgets

In [2]:
# Requirements
# - PyTorch
# - Kagglehub
# - Pandas
# - Datasets

from datasets import load_dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import re
import torch
import numpy as np
import math
from torch.utils.data import DataLoader, TensorDataset, random_split


from ray import tune
from ray.tune.schedulers import ASHAScheduler
import ray

from filelock import FileLock
import os


In [3]:

def get_two_resp(convo):
        responses = re.split(" '[ \n]*' ", convo)

        if len(responses) >= 2:
            return responses[0], responses[1]
        else:
             return None, None

def clean_str(str):
     str = re.sub("[^ a-zA-Z]+", "", str)
     str = re.sub("[ ]+", " ", str)
     str = str.strip()
     str = str.lower()
     str = re.sub(" [ ]+", " ", str)
     return str


In [4]:

def clean(df):
    dialog = df["dialog"]

    q_a = np.empty((len(dialog), 2), dtype=np.dtypes.StringDType) # Numpy array where first col is question and second is the answer

    i = 0
    for convo in dialog:
        first_q, first_a = get_two_resp(convo)
        if first_q == None:
              continue

        clean_q = clean_str(first_q)
        clean_a = clean_str(first_a)

        if len(clean_q) > 0 and len(clean_a) > 0:
            q_a[i][0] = clean_q
            q_a[i][1] = clean_a
            i = i + 1


    # Remove empty rows at the end
    q_a = q_a[0:i]

    return q_a




In [5]:

# Tokenize the data

def validate_text(text):
    # Check for capital letters
    if bool(re.search(r'[A-Z]', text)):
        raise ValueError("Text must be all lowercase")
    if bool(re.search(r'[^a-z ]', text)):
        raise ValueError("Text must be all lowercase letters and spaces")
    if bool(re.search(r'  ', text)):
        raise ValueError("Text must not include double spaces")

    if len(text) <= 1:
        print(text)
        print("about to err")
    if text[0] == " " or text[len(text) - 1] == " ":
        print(text)
        print("is text")
        raise ValueError("Text must be stripped")


def load_token_dataset(dataset):
    start_token = 0
    end_token = 1
    token_map = {"PAD":0, "START_TOKEN":1, "END_TOKEN":2} # Converts token to id quickly
    id_map = ["PAD", "START_TOKEN", "END_TOKEN"] # Index is the token id. Converts id to string token


    token_id = 3 # Counter for new ID
    max_tokens = -1 # Track the most number of tokens needed for question or answer in the database
    for row in dataset:
        # Get the text from the question and answer
        questions = row[0]
        answers = row[1]
        validate_text(questions)
        validate_text(answers)

        question_strs = questions.split(" ")
        answer_strs = answers.split(" ")

        if len(question_strs) > max_tokens:
            max_tokens = len(question_strs)
        if len(answer_strs) > max_tokens:
            max_tokens = len(answer_strs)

        # Add each new unique question to token_map
        for t in question_strs:
            if token_map.get(t) == None:
                token_map[t] = token_id
                id_map.append(t)
                token_id = token_id + 1

        # Add each new unique answer to token_map
        for t in answer_strs:
            if token_map.get(t) == None:
                token_map[t] = token_id
                id_map.append(t)
                token_id = token_id + 1

    return token_map, id_map, max_tokens


In [6]:

# Tokenizes a string and pads with zeros to the longest sentence length plus two (for start/end tokens)
def tokenize(text, token_map, max_tokens_len):
    token_strs = text.lower().split(" ")
    if len(token_strs) > max_tokens_len + 2:
        raise ValueError("Text may not have more tokens than the max_tokens_len + 2")
    tokens = np.zeros(max_tokens_len + 2, dtype=np.int64) # Include two extra tokens for start and end
    tokens[0] = token_map["START_TOKEN"]
    i = 1
    for t in token_strs:
        if token_map.get(t) != None:
            tokens[i] = token_map.get(t)
            i = i + 1
        else:
            pass # Skip unknown inputs


    tokens[i] = token_map["END_TOKEN"]

    return np.array(tokens)

def tokenize_dataset(q_a, token_map, longest_sentence):
    rows, cols = np.shape(q_a)
    tokenized_inputs = np.zeros((rows, cols, longest_sentence + 2), dtype=np.int64) # Third dimen is longest_sentence + 2 to include start/end tokens

    for i, row in enumerate(q_a):
        tokenized_inputs[i][0] = tokenize(row[0], token_map, longest_sentence) # Returns a numpy array
        tokenized_inputs[i][1] = tokenize(row[1], token_map, longest_sentence) # Returns a numpy array

    return tokenized_inputs



In [7]:


# Positional encoding - gives the model the original order of inputs

def get_angles(pos, i, d_model):
    angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
    return pos * angle_rates

def positional_encoding(position, d_model):
    angle_rads = get_angles(np.arange(position)[:, np.newaxis], np.arange(d_model)[np.newaxis, :], d_model)
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2]) #for even positions using sin()
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2]) #for odd positions using cos()
    pos_encoding = angle_rads[np.newaxis,:]
    return torch.from_numpy(pos_encoding)

In [8]:

# Mask tokens. Any tokens beyond "END_TOKEN" will be masked
def create_padding_mask(seq):
    seq = torch.eq(seq, 0).to(torch.float32)
    return seq[:, torch.newaxis, torch.newaxis, :]

# Mask future tokens
def create_lookahead_mask(size):
    return torch.triu(torch.ones((size, size)), 1)


def scaled_dot_product_attention(q, k, v, mask=None):
    matmul_qk = torch.matmul(q, k.transpose(-2, -1))
    dk = k.size(-1)
    scaled_attention_logits = matmul_qk / math.sqrt(dk)
    if mask is not None:
        scaled_attention_logits += (mask * -1e9)  # -1e9 ~ (-INFINITY) => where ever mask is set, make its logit value close to -INF
    attention_weights = torch.softmax(scaled_attention_logits, dim=-1)
    output = torch.matmul(attention_weights, v)

    return output, attention_weights


In [9]:

class MultiHeadAttentionClass(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0

        self.depth = d_model // num_heads
        self.wq = torch.nn.Linear(d_model, d_model)
        self.wk = torch.nn.Linear(d_model, d_model)
        self.wv = torch.nn.Linear(d_model, d_model)
        self.dense = torch.nn.Linear(d_model, d_model)

    # Resize into (batch_size, num_heads, seq_len, depth)
    def split_heads(self, x, batch_size):
        x = torch.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        x = torch.permute(x, (0, 2, 1, 3)) # Tf used transpose
        return x

    def forward(self, v, k, q, mask):
        batch_size = q.shape[0]
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        scaled_attention, attention_weights = scaled_dot_product_attention(q, k, v, mask)
        scaled_attention = torch.permute(scaled_attention, (0, 2, 1, 3))
        concat_attention = torch.reshape(scaled_attention, (batch_size, -1, self.d_model))
        output = self.dense(concat_attention)

        return output, attention_weights


In [10]:

def point_wise_feed_forward_network(d_model, dff):
    return torch.nn.Sequential(torch.nn.Linear(d_model, dff), torch.nn.ReLU(), torch.nn.Linear(dff, d_model))

class EncoderLayer(torch.nn.Module):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttentionClass(d_model, num_heads)
        self.layerNorm1 = torch.nn.LayerNorm(d_model, eps=1e-6)
        self.layerNorm2 = torch.nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = torch.nn.Dropout(rate)
        self.dropout2 = torch.nn.Dropout(rate)
        self.ffn = point_wise_feed_forward_network(d_model, dff)

    def forward(self, x, mask):
        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        out1 = self.layerNorm1(x + attn_output)

        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out)
        out2 = self.layerNorm2(out1 + ffn_out)

        return out2

class Encoder(torch.nn.Module):
    def __init__(self, device, num_layers, d_model, num_heads, dff, input_vocab_size, max_positional_encoding, rate=0.1):
        super(Encoder, self).__init__()
        self.num_layers = num_layers
        self.d_model = d_model
        self.embedding = torch.nn.Embedding(input_vocab_size, d_model)
        self.positional_encoding = positional_encoding(max_positional_encoding, d_model).to(device)
        self.encoder_layers = torch.nn.ModuleList([EncoderLayer(d_model, num_heads, dff, rate) for i in range(num_layers)])
        self.dropout = torch.nn.Dropout(rate)

    def forward(self, x, mask):
        sequence_length = x.shape[1]

        x = self.embedding(x)
        x *= math.sqrt(torch.tensor(self.d_model)) # Done in research
        y = self.positional_encoding[:, :sequence_length, :]
        x += y
        x = self.dropout(x)

        for i, el in enumerate(self.encoder_layers):
            x = el(x, mask)

        return x # (batch_size, input_seq_len, d_model)


In [11]:

class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttentionClass(d_model, num_heads)
        self.mha2 = MultiHeadAttentionClass(d_model, num_heads)

        self.dropout1 = torch.nn.Dropout(rate)
        self.dropout2 = torch.nn.Dropout(rate)
        self.dropout3 = torch.nn.Dropout(rate)

        self.layerNorm1 = torch.nn.LayerNorm(d_model, eps=1e-6)
        self.layerNorm2 = torch.nn.LayerNorm(d_model, eps=1e-6)
        self.layerNorm3 = torch.nn.LayerNorm(d_model, eps=1e-6)

        self.fnn = point_wise_feed_forward_network(d_model, dff)

    def forward(self, x, enc_layer, look_ahead_mask, padding_mask):
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1)
        out1 = self.layerNorm1(attn1 + x)

        attn2, attn_weights_block2 = self.mha2(enc_layer, enc_layer, out1, padding_mask)
        attn2 = self.dropout2(attn2)
        out2 = self.layerNorm2(attn2 + out1)

        ffn_out = self.fnn(out2)
        ffn_out = self.dropout3(ffn_out)
        out3 = self.layerNorm3(ffn_out + out2)

        return out3, attn_weights_block1, attn_weights_block2


class Decoder(torch.nn.Module):
    def __init__(self, device, num_layers, d_model, num_heads, dff, target_vocab_size, max_positional_encoding, rate=0.1):
        super().__init__()
        self.num_layers = num_layers
        self.d_model = d_model
        self.embedding = torch.nn.Embedding(target_vocab_size, d_model)
        self.positional_encoding = positional_encoding(max_positional_encoding, d_model).to(device)
        self.decoder_layers = torch.nn.ModuleList([DecoderLayer(d_model, num_heads, dff, rate) for i in range(num_layers)])
        self.dropout = torch.nn.Dropout(rate)

    def forward(self, x, encoding_output, look_ahead_mask, padding_mask):
        sequence_length = x.shape[1]
        attention_weights = {}
        x = self.embedding(x)
        x *= math.sqrt(self.d_model)
        x += self.positional_encoding[:, :sequence_length, :]
        x = self.dropout(x)

        for i, dl in enumerate(self.decoder_layers):
            x, attn_weights_block1, attn_weights_block2 = dl(x, encoding_output, look_ahead_mask, padding_mask)
            attention_weights['decoder_layer{}_block1'.format(i+1)] = attn_weights_block1
            attention_weights['decoder_layer{}_block2'.format(i+1)] = attn_weights_block2

        return x, attention_weights


In [12]:

class Transformer(torch.nn.Module):
    def __init__(self, device, num_layers, d_model, num_heads, dff, input_vocab_size, target_vocab_size, pe_input, pe_target, rate=0.1):
        super().__init__()
        self.encoder = Encoder(device, num_layers, d_model, num_heads, dff, input_vocab_size, pe_input, rate)
        self.decoder = Decoder(device, num_layers, d_model, num_heads, dff, target_vocab_size, pe_target, rate)
        self.nn = torch.nn.Linear(d_model, target_vocab_size)

    def forward(self, inp, tar, enc_padding_mask, look_ahead_mask, dec_padding_mask):
        enc_output = self.encoder(inp, enc_padding_mask)
        dec_output, attn_weights = self.decoder(tar, enc_output, look_ahead_mask, dec_padding_mask)
        dec_output = self.nn(dec_output)

        return dec_output, attn_weights


In [13]:
def load_data(data_dir="./data"):
    with FileLock(os.path.expanduser("~/.data.lock")):
        # Todo - update this to download locally
        train_df = kagglehub.dataset_load(
            KaggleDatasetAdapter.PANDAS,
            "thedevastator/dailydialog-unlock-the-conversation-potential-in",
            "train.csv",
            )

        test_df = kagglehub.dataset_load(
            KaggleDatasetAdapter.PANDAS,
            "thedevastator/dailydialog-unlock-the-conversation-potential-in",
            "train.csv",
            )

        train_qa_dataset = clean(train_df)
        train_token_map, train_id_map, train_max_tokens = load_token_dataset(train_qa_dataset)
        train_tokenized_dataset = tokenize_dataset(train_qa_dataset, train_token_map, train_max_tokens)
        train_tokenized_dataset = torch.from_numpy(train_tokenized_dataset)


        test_qa_dataset = clean(test_df)
        test_token_map, test_id_map, test_max_tokens = load_token_dataset(test_qa_dataset)
        test_tokenized_dataset = tokenize_dataset(test_qa_dataset, test_token_map, test_max_tokens)
        test_tokenized_dataset = torch.from_numpy(test_tokenized_dataset)


    return train_tokenized_dataset, test_tokenized_dataset, train_token_map, test_token_map, train_id_map, test_id_map

def create_dataloaders(trainset, batch_size, num_workers=8):
    """Create train/val splits and dataloaders."""
    train_size = int(len(trainset) * 0.8)

    src = trainset[:, 0, :].long()
    tgt_full = trainset[:, 1, :].long()
    tgt_in = tgt_full[:, :-1]
    tgt_out = tgt_full[:, 1:]

    trainset = TensorDataset(src, tgt_in, tgt_out)
    train_subset, val_subset = random_split(
        trainset, [train_size, len(trainset) - train_size])

    train_loader = torch.utils.data.DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )
    val_loader = torch.utils.data.DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    pe_input = src.size(1)
    pe_target = tgt_in.size(1)

    return train_loader, val_loader, pe_input, pe_target

In [14]:

def train_model(
    config
):

    train_loader=config["train_loader"],
    val_loader=config["val_loader"],
    pe_input=config["pe_input"],
    pe_target=config["pe_target"],
    vocab_size=config["vocab_size"],
    pad_token_id=config["pad_token_id"],
    batch_size=config["batch_size"],
    epochs=config["epochs"] if config["epochs"] else 5,
    learning_rate=config["learning_rate"] if config["learning_rate"] else 3e-4,
    val_split=config["val_split"] if config["val_split"] else 0.1,
    d_model=config["d_model"] if config["d_model"] else 256,
    num_heads=config["num_heads"] if config["num_heads"] else 8,
    num_layers=config["num_layers"] if config["num_layers"] else 2,
    dff=config["dff"] if config["dff"] else 512,
    dropout=0.1,
    device = config["device"] if config["device"] else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device is", device)

    # src = tokenized_dataset[:, 0, :].long()
    # tgt_full = tokenized_dataset[:, 1, :].long()
    # tgt_in = tgt_full[:, :-1]
    # tgt_out = tgt_full[:, 1:]

    # dataset = TensorDataset(src, tgt_in, tgt_out)
    # val_size = int(len(dataset) * val_split)
    # train_size = len(dataset) - val_size
    # train_ds, val_ds = random_split(dataset, [train_size, val_size])

    # train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    # val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False) if val_size > 0 else None

    model = Transformer(
        device,
        input_vocab_size=vocab_size,
        target_vocab_size=vocab_size,
        # pe_input=src.size(1),
        # pe_target=tgt_in.size(1),
        pe_input=pe_input,
        pe_target=pe_target,
        d_model=d_model,
        num_heads=num_heads,
        num_layers=num_layers,
        dff=dff,
        rate=dropout,
    ).to(device)

    criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_token_id)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0

        for batch_src, batch_tgt_in, batch_tgt_out in train_loader:
            batch_src = batch_src.to(device)
            batch_tgt_in = batch_tgt_in.to(device)
            batch_tgt_out = batch_tgt_out.to(device)

            enc_padding_mask = create_padding_mask(batch_src).to(device)
            dec_padding_mask = create_padding_mask(batch_src).to(device)
            look_ahead_mask = create_lookahead_mask(batch_tgt_in.size(1)).to(device)
            dec_target_padding_mask = create_padding_mask(batch_tgt_in).to(device)
            combined_mask = torch.maximum(
                dec_target_padding_mask,
                look_ahead_mask.unsqueeze(0).unsqueeze(0),
            )

            optimizer.zero_grad()
            logits, _ = model(
                inp=batch_src,
                tar=batch_tgt_in,
                enc_padding_mask=enc_padding_mask,
                look_ahead_mask=combined_mask,
                dec_padding_mask=dec_padding_mask,
            )

            loss = criterion(logits.reshape(-1, vocab_size), batch_tgt_out.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / max(1, len(train_loader))

        if val_loader is not None:
            model.eval()
            total_val_loss = 0.0
            with torch.no_grad():
                for batch_src, batch_tgt_in, batch_tgt_out in val_loader:
                    batch_src = batch_src.to(device)
                    batch_tgt_in = batch_tgt_in.to(device)
                    batch_tgt_out = batch_tgt_out.to(device)

                    enc_padding_mask = create_padding_mask(batch_src).to(device)
                    dec_padding_mask = create_padding_mask(batch_src).to(device)
                    look_ahead_mask = create_lookahead_mask(batch_tgt_in.size(1)).to(device)
                    dec_target_padding_mask = create_padding_mask(batch_tgt_in).to(device)
                    combined_mask = torch.maximum(
                        dec_target_padding_mask,
                        look_ahead_mask.unsqueeze(0).unsqueeze(0),
                    )

                    logits, _ = model(
                        inp=batch_src,
                        tar=batch_tgt_in,
                        enc_padding_mask=enc_padding_mask,
                        look_ahead_mask=combined_mask,
                        dec_padding_mask=dec_padding_mask,
                    )
                    loss = criterion(logits.reshape(-1, vocab_size), batch_tgt_out.reshape(-1))
                    total_val_loss += loss.item()

            avg_val_loss = total_val_loss / max(1, len(val_loader))
            print(f"Epoch {epoch}/{epochs} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}")
        else:
            print(f"Epoch {epoch}/{epochs} | train_loss={avg_train_loss:.4f}")

    return model



# Driver code

In [15]:



# # # Set the path to the file you'd like to load
# # file_path = "test.csv"

# # # Load the latest version
# # df = kagglehub.load_dataset(
# # KaggleDatasetAdapter.PANDAS,
# # "thedevastator/dailydialog-unlock-the-conversation-potential-in",
# # file_path,
# # )


# num_layers = 2
# d_model = 256
# dff = 512
# num_heads = 8
# dropout_rate = 0.1
# batch_size = 128
# epochs = 10
# learning_rate = 3e-4

# tok_train, tok_test, train_token_map, test_token_map, train_id_map, test_id_map = load_data()
# train_loader, val_loader, pe_input, pe_target = create_dataloaders(tok_train, batch_size, num_workers=8)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# model = train_model_on_tokenized_dataset(
#     train_loader=train_loader,
#     val_loader=val_loader,
#     pe_input=pe_input,
#     pe_target=pe_target,
#     vocab_size=len(train_id_map),
#     pad_token_id=train_token_map["PAD"],
#     batch_size=batch_size,
#     epochs=epochs,
#     learning_rate=learning_rate,
#     d_model=d_model,
#     num_heads=num_heads,
#     num_layers=num_layers,
#     dff=dff,
#     dropout=dropout_rate,
# )


In [16]:
'''
num layers =


    train_loader=config["train_loader"],
    val_loader=config["val_loader"],
    pe_input=config["pe_input"],
    pe_target=config["pe_target"],
    vocab_size=config["vocab_size"],
    pad_token_id=config["pad_token_id"],
    batch_size=config["batch_size"],
#     batch_size=batch_size,
#     epochs=epochs,
#     learning_rate=learning_rate,
#     d_model=d_model,
#     num_heads=num_heads,
#     num_layers=num_layers,
#     dff=dff,
#     dropout=dropout_rate,

d_model % num heads = 0
'''

batch_size = 128

tok_train, tok_test, train_token_map, test_token_map, train_id_map, test_id_map = load_data()
train_loader, val_loader, pe_input, pe_target = create_dataloaders(tok_train, batch_size, num_workers=8)




config = {
    "train_loader": train_loader,
    "val_loader": val_loader,
    "pe_input": pe_input,
    "pe_target": pe_target,
    "vocab_size": len(train_id_map),
    "pad_token_id": train_token_map["PAD"],

    "num_layers": tune.choice([2, 4, 8, 16]),
    "d_model": tune.sample_from(lambda _: 2**np.random.randint(7, 9)),
    "dff": tune.sample_from(lambda _: 2**np.random.randint(9, 11)),
    "num_heads": tune.choice([2, 4, 8]),
    "dropout_rate": tune.sample_from(lambda _: 0.05 ** np.random.randint(1, 4)),
    "batch_size": batch_size,
    "max_num_epochs": 10,
    "num_trials": 100,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "learning_rate": tune.sample_from(lambda _: np.random.randint(1, 9)*10**np.random.randint(-4, -2)),
}

Using Colab cache for faster access to the 'dailydialog-unlock-the-conversation-potential-in' dataset.
Using Colab cache for faster access to the 'dailydialog-unlock-the-conversation-potential-in' dataset.


In [ ]:
scheduler = ASHAScheduler(
    time_attr="training_iteration",
    max_t=config["max_num_epochs"],
    grace_period=1,
    reduction_factor=2)

tuner = tune.Tuner(
    tune.with_resources(
        tune.with_parameters(train_model),
        resources={"cpu": 1, "gpu": 1}
    ),
    tune_config=tune.TuneConfig(
        metric="loss",
        mode="min",
        scheduler=scheduler,
        num_samples=config["num_trials"],
    ),
    param_space=config,
)
results = tuner.fit()

best_result = results.get_best_result("loss", "min")

print(f"Best trial config: {best_result.config}")
print(f"Best trial final validation loss: {best_result.metrics['loss']}")
print(f"Best trial final validation accuracy: {best_result.metrics['accuracy']}")

2026-04-02 15:26:08,866	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-04-02 15:26:12,201	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `Tuner(...)`.


+--------------------------------------------------------------------+
| Configuration for experiment     train_model_2026-04-02_15-25-59   |
+--------------------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator             |
| Scheduler                        AsyncHyperBandScheduler           |
| Number of trials                 100                               |
+--------------------------------------------------------------------+

View detailed results here: /root/ray_results/train_model_2026-04-02_15-25-59
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-04-02_15-25-59_043305_14188/artifacts/2026-04-02_15-26-12/train_model_2026-04-02_15-25-59/driver_artifacts`
